<a href="https://colab.research.google.com/github/elkinkazan/itis-2025-cv-course/blob/main/Feature_Matching_Autograder_Iliasova.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🧪 Home Assignment: Feature Detection & Matching — Autograder (Colab)





In [ ]:
# === ОБЯЗАТЕЛЬНО ЗАПОЛНИТЬ ===
full_name = "Эльвира Ильясова"     # например: "Тощев Александр"
student_group = "11-451"      # например: "208"
assignment_id = "HLP_AvoidableBias"
assert full_name != "Фамилия Имя", "Заполните full_name"
assert student_group != "Группа", "Заполните student_group"
print("✔ Student Info OK")

print("Student:", full_name)

✔ Student Info OK
Student: Эльвира Ильясова


In [ ]:

# Colab / Jupyter-ready cell
# Home assignment auto-checker: Feature detection & matching
# Usage: student edits cells marked with "=== YOUR CODE HERE ==="
# Instructor: set DUE_DATE (ISO string) and LATE_WINDOW_DAYS

# 1) Настройки (инструктор)
from datetime import datetime, timezone, timedelta
import json
import os
import sys
from pathlib import Path

from datetime import datetime, timezone, timedelta

# Установите окна приёма (пример):

start_at_iso = "2025-10-07T09:00-04:00"  #@param {type:"string"}
due_at_iso   = "2025-10-21T23:59-04:00"  #@param {type:"string"}
start_dt = datetime.fromisoformat(start_at_iso)
due_dt   = datetime.fromisoformat(due_at_iso)
# Для протокола: время сдачи берём текущее (можно заменить на mtime файла)
import os
from datetime import datetime, timezone

# 📅 Add submission date based on file modification time
try:
    nb_path = __file__ if "__file__" in globals() else "Feature_Matching_Autograder.ipynb"
    mtime = os.path.getmtime(nb_path)
    submission_dt = datetime.fromtimestamp(mtime, tz=timezone.utc)
except Exception:
    submission_dt = datetime.utcnow().replace(tzinfo=timezone.utc)



# 2) Установим зависимости (OpenCV, scikit-image и т.д.)
# В Colab: раскомментируйте и выполните
# !pip install opencv-contrib-python scikit-image numpy pytest

import numpy as np
import cv2
from skimage.metrics import structural_similarity as ssim
import tempfile
import math
import time

# 3) Helper: читает время сдачи. В Colab студент указывает строкой (или можно взять mtime файла)
def parse_iso_datetime(s):
    # robust parse of ISO-like strings
    return datetime.fromisoformat(s)



# 4) Задачи (инструктор описывает, студент реализует)

print("Due:", due_dt, "| Submission time detected:", submission_dt.isoformat())



Due: 2025-10-21 23:59:00-04:00 | Submission time detected: 2025-10-14T13:52:53.031962+00:00


/tmp/ipython-input-295019031.py:31: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  submission_dt = datetime.utcnow().replace(tzinfo=timezone.utc)


In [ ]:

# === STUDENT: Implement these functions ===
# Place clear "YOUR CODE HERE" markers where students must code.

def detect_and_describe(image):
    """
    Detect keypoints and compute descriptors.
    Return: keypoints (list of cv2.KeyPoint), descriptors (np.ndarray)
    Student task: choose detector (ORB/SIFT) and return.
    """
    # === YOUR CODE HERE ===
    # Example (allowed as reference): ORB detector
    if image.ndim == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image
    orb = cv2.ORB_create(nfeatures=2000, scaleFactor=1.2, nlevels=8, edgeThreshold=15, firstLevel=0, WTA_K=2, scoreType=cv2.ORB_HARRIS_SCORE, patchSize=31)

    kps, desc = orb.detectAndCompute(gray, None)
    return kps, desc

def match_descriptors(desc1, desc2, ratio_test=0.75):
    """
    Match descriptors between two images and apply ratio test (Lowe).
    Return: list of cv2.DMatch objects (good matches)
    """
    # === YOUR CODE HERE ===
    # Example for ORB (binary descriptors): use BFMatcher with NORM_HAMMING
    if desc1 is None or desc2 is None or len(desc1) == 0 or len(desc2) == 0:
        return []
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    raw = bf.knnMatch(desc1, desc2, k=2)
    good = []
    for m_n in raw:
        if len(m_n) != 2:
            continue
        m, n = m_n
        if m.distance < ratio_test * n.distance:
            good.append(m)
    return good

def ransac_filter_matches(kp1, kp2, matches, reproj_thresh=3.0):
    """
    Use RANSAC to estimate homography and return inlier matches and the homography.
    """
    if len(matches) < 4:
        return [], None, None
    ptsA = np.float32([kp1[m.queryIdx].pt for m in matches])
    ptsB = np.float32([kp2[m.trainIdx].pt for m in matches])
    # === YOUR CODE HERE ===
    H, mask = cv2.findHomography(ptsA, ptsB, cv2.RANSAC, ransacReprojThreshold=reproj_thresh)

    if mask is None:
        return [], H, None
    mask = mask.ravel().tolist()
    inliers = [m for m, inl in zip(matches, mask) if inl]
    return inliers, H, mask

# =========== End student-implementable functions ===========

# 5) Metrics / tests (autograder)
def evaluate_pair(img1, img2, gt_H=None):
    """
    Full pipeline: detect -> describe -> match -> ransac -> metrics
    Returns dictionary with metrics
    """
    kp1, desc1 = detect_and_describe(img1)
    kp2, desc2 = detect_and_describe(img2)
    matches = match_descriptors(desc1, desc2)
    inliers, H, mask = ransac_filter_matches(kp1, kp2, matches)
    metrics = {
        "num_kp1": len(kp1),
        "num_kp2": len(kp2),
        "num_matches": len(matches),
        "num_inliers": len(inliers),
        "ratio_inliers_matches": (len(inliers) / len(matches)) if len(matches)>0 else 0.0,
        "homography_found": H is not None
    }
    # optional: geometric verification via reprojection if gt_H provided
    return metrics, {"kp1":kp1, "kp2":kp2, "matches":matches, "inliers":inliers, "H":H}

# 6) Rubric (instructor can tune weights)
RUBRIC = {
    "feature_detection_working": 30,   # returns keypoints & descriptors
    "matching_correctness": 30,       # good matches found (thresholded)
    "ransac_inliers": 40,            # sufficient inliers & homography estimation
    "code_quality_and_docs": 20      # style, comments, reproducibility
}
TOTAL_POINTS = sum(RUBRIC.values())

# 7) Example runner: load sample images (instructor: replace with real dataset)
def load_sample_images():
    # Two example images included with OpenCV samples could be used.
    # Instructor can mount Google Drive and point to dataset folder.
    # For demo, create synthetic transform or use opencv data if available.
    # Here: create a small synthetic test (translation + rotation) on an image if not provided.
    img_path = None
    fallback = True
    if fallback:
        # generate synthetic image with simple shapes
        base = np.zeros((300,400,3), dtype=np.uint8)
        cv2.circle(base, (200,150), 60, (255,255,255), -1)
        cv2.rectangle(base, (50,50), (120,120), (255,255,255), -1)
        M = cv2.getRotationMatrix2D((200,150), 10, 1.0)
        img2 = cv2.warpAffine(base, M, (400,300))
        return base, img2
    else:
        # example: cv2.imread from Drive
        return None, None

img1, img2 = load_sample_images()

# 8) Run evaluation on the example pair
metrics, aux = evaluate_pair(img1, img2)
print("Metrics:", metrics)

# 9) Convert metrics to score per rubric (simple heuristics — tune as needed)
def score_from_metrics(metrics):
    score = 0.0
    # feature_detection_working
    if metrics["num_kp1"]>10 and metrics["num_kp2"]>10:
        score += RUBRIC["feature_detection_working"]
    # matching_correctness: reward for > N matches
    if metrics["num_matches"] >= 10:
        score += RUBRIC["matching_correctness"] * min(1.0, metrics["num_matches"]/50.0)
    else:
        score += RUBRIC["matching_correctness"] * (metrics["num_matches"]/10.0) * 0.5
    # ransac_inliers: reward for ratio and absolute inliers
    inlier_score = 0.0
    if metrics["num_inliers"] >= 8:
        inlier_score = 1.0
    else:
        inlier_score = metrics["ratio_inliers_matches"] * 0.6
    score += RUBRIC["ransac_inliers"] * inlier_score
    # code quality: placeholder — instructor/manual
    # default give half, instructor can edit
    return max(0.0, min(TOTAL_POINTS, score))

raw_score = score_from_metrics(metrics)






Metrics: {'num_kp1': 169, 'num_kp2': 231, 'num_matches': 52, 'num_inliers': 32, 'ratio_inliers_matches': 0.6153846153846154, 'homography_found': True}


In [ ]:
import json

# применяем штраф
try:
    pf = penalty_fraction(start_dt, due_dt, submission_dt)
except NameError:
    from datetime import timezone
    pf = 0.0
# ✅ Итоговый результат


final_score = max(0.0, raw_score * (1.0 - min(1.0, pf)))

print(f"Сырой балл: {raw_score}/{100}")
print(f"Штраф (доля): {pf:.4f}")
print(f"Итоговый балл после штрафа: {final_score:.2f}/{100}")

# Последняя строка — JSON, который читает harness
final = {
    "name": full_name,
    "group": student_group,
    "assignment": assignment_id,
    "score": float(final_score)
}

Сырой балл: 100.0/100
Штраф (доля): 0.0000
Итоговый балл после штрафа: 100.00/100


In [ ]:
print(json.dumps(final, ensure_ascii=False))

{"name": "Эльвира Ильясова", "group": "11-451", "assignment": "HLP_AvoidableBias", "score": 100.0}
